In [ ]:
!pip install transformers[torch] datasets scikit-learn matplotlib seaborn

In [ ]:
import pandas as pd
import numpy as np
import torch
import os
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset

# Pastikan menggunakan GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Menggunakan device: {device.upper()}")

# Konfigurasi Eksperimen
apps = ['satusehat', 'signal', 'ikd']
models_to_test = {
    'IndoBERT': 'indobenchmark/indobert-base-p1',
    'mBERT': 'bert-base-multilingual-uncased'
}

hasil_evaluasi = []

def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='weighted', zero_division=0)
    acc = accuracy_score(labels, preds)
    return {'accuracy': acc, 'f1': f1, 'precision': precision, 'recall': recall}

for app in apps:
    print(f"\n{'='*50}\nMEMPROSES APLIKASI: {app.upper()}\n{'='*50}")

    # 1. Load Data
    path_file = f"/content/data/{app}_cleaned_reviews.csv"
    df = pd.read_csv(path_file)

    df['label'] = df['sentiment'].apply(lambda x: 1 if x == 'Positif' else 0)
    df = df.dropna(subset=['cleaned_content'])

    # Split Data
    train_texts, test_texts, train_labels, test_labels = train_test_split(
        df['cleaned_content'].tolist(),
        df['label'].tolist(),
        test_size=0.2,
        random_state=42,
        stratify=df['label'].tolist()
    )

    for model_name, model_path in models_to_test.items():
        print(f"\n---> Mulai Fine-Tuning: {model_name} pada {app.upper()} <---")

        # 2. Load Tokenizer & Model
        tokenizer = AutoTokenizer.from_pretrained(model_path)
        model = AutoModelForSequenceClassification.from_pretrained(model_path, num_labels=2).to(device)

        train_encodings = tokenizer(train_texts, truncation=True, padding=True, max_length=128)
        test_encodings = tokenizer(test_texts, truncation=True, padding=True, max_length=128)

        class SentDataset(torch.utils.data.Dataset):
            def __init__(self, encodings, labels):
                self.encodings = encodings
                self.labels = labels
            def __getitem__(self, idx):
                item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
                item['labels'] = torch.tensor(self.labels[idx])
                return item
            def __len__(self):
                return len(self.labels)

        train_dataset = SentDataset(train_encodings, train_labels)
        test_dataset = SentDataset(test_encodings, test_labels)

        # 3. Konfigurasi Training (ERROR SUDAH DIPERBAIKI DI SINI)
        training_args = TrainingArguments(
            output_dir=f'./results_{model_name}_{app}',
            num_train_epochs=3,
            per_device_train_batch_size=16,
            per_device_eval_batch_size=32,
            eval_strategy="epoch",  # <-- INI YANG DIUBAH
            save_strategy="epoch",
            logging_dir='./logs',
            load_best_model_at_end=True,
            fp16=True
        )

        trainer = Trainer(
            model=model,
            args=training_args,
            train_dataset=train_dataset,
            eval_dataset=test_dataset,
            compute_metrics=compute_metrics
        )

        # 4. Training
        trainer.train()

        # 5. Evaluasi
        eval_result = trainer.evaluate()
        print(f"Hasil {model_name} - {app.upper()}: {eval_result}")

        hasil_evaluasi.append({
            'Aplikasi': app.upper(),
            'Model': model_name,
            'Accuracy': round(eval_result['eval_accuracy'] * 100, 2),
            'Precision': round(eval_result['eval_precision'] * 100, 2),
            'Recall': round(eval_result['eval_recall'] * 100, 2),
            'F1-Score': round(eval_result['eval_f1'] * 100, 2)
        })

        # 6. Confusion Matrix
        predictions = trainer.predict(test_dataset)
        preds = predictions.predictions.argmax(-1)
        cm = confusion_matrix(test_labels, preds)

        plt.figure(figsize=(6, 4))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Negatif', 'Positif'], yticklabels=['Negatif', 'Positif'])
        plt.xlabel('Tebakan Model (Predicted)')
        plt.ylabel('Kenyataan (Actual)')
        plt.title(f'Confusion Matrix: {model_name} - {app.upper()}')

        nama_gambar_cm = f"CM_{model_name}_{app}.png"
        plt.savefig(nama_gambar_cm, dpi=300, bbox_inches='tight')
        plt.close()

# 7. Simpan Hasil Akhir
df_hasil = pd.DataFrame(hasil_evaluasi)
df_hasil.to_excel('Tabel_Performa_Benchmarking.xlsx', index=False)
print("\n" + "="*50)
print("SELURUH PROSES SELESAI!")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Angka yang diambil dari hasil Colab-mu
cm = np.array([[569, 73], [95, 212]])

# 1. Pengaturan Font ala Paper Akademik
plt.rcParams["font.family"] = "serif"
plt.rcParams["font.serif"] = ["Times New Roman"]
plt.rcParams["font.size"] = 12

# 2. Membuat Label Text yang Lebih Informatif
group_names = ['True Negative (TN)', 'False Positive (FP)', 'False Negative (FN)', 'True Positive (TP)']
group_counts = ["{0:0.0f}".format(value) for value in cm.flatten()]
labels = [f"{v1}\n{v2}" for v1, v2 in zip(group_names, group_counts)]
labels = np.asarray(labels).reshape(2,2)

# 3. Menggambar Heatmap
plt.figure(figsize=(6, 5))
ax = sns.heatmap(cm, annot=labels, fmt='', cmap='Blues',
                 xticklabels=['Negatif', 'Positif'],
                 yticklabels=['Negatif', 'Positif'],
                 cbar=False, # Hilangkan colorbar di samping agar lebih rapi
                 annot_kws={"size": 14, "weight": "bold"})

# 4. Merapikan Judul dan Sumbu
plt.xlabel('Tebakan Model (Predicted)', fontweight='bold', labelpad=15)
plt.ylabel('Kenyataan (Actual)', fontweight='bold', labelpad=15)
plt.title('Confusion Matrix: mBERT - SIGNAL', fontweight='bold', pad=20, fontsize=16)

# 5. Simpan dengan resolusi sangat tinggi (600 DPI)
nama_file = 'CM_mBERT_SIGNAL_PaperReady.png'
plt.savefig(nama_file, dpi=600, bbox_inches='tight')
plt.show()

print(f"Gambar siap masuk paper! Silakan download {nama_file}")